In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect("../recon_view.duckdb", read_only=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## Сырые файлы

- `payment_engine_log.csv` — лог платёжного движка, источник истины, все провайдеры
- `paypal_us_activity_20260601_20260703.csv` — выписка PayPal US
- `paypal_eu_activity_20260601_20260703.csv` — выписка PayPal EU (`;`, cp1252)
- `adyen_payment_accounting_20260601_20260703.csv` — отчёт Adyen payment accounting
- `dlocal_transactions_20260601_20260703.csv` — экспорт транзакций dLocal
- `google_play_earnings_202606.csv` — отчёт Google Play за июнь
- `google_play_earnings_202607_partial.csv` — отчёт Google Play, 1–3 июля
- `fx_rates.csv` — дневные курсы к USD
- `fee_schedule.csv` — контрактные комиссии провайдеров

In [2]:
con.sql("""
    select table_schema, table_name
    from information_schema.tables
    order by 1, 2
""").df()

,table_schema,table_name
0,analysis,profile_duplicates
1,analysis,profile_summary
2,analysis,profile_values
3,raw,raw_adyen_payment_accounting
4,raw,raw_dlocal_transactions
5,raw,raw_fee_schedule
6,raw,raw_fx_rates
7,raw,raw_google_play_earnings_202606
8,raw,raw_google_play_earnings_202607_partial
9,raw,raw_payment_engine_log


## Логи движка

In [3]:
log = con.sql("select * from raw.raw_payment_engine_log").df()
print(log.shape)
log.head()

(7988, 15)


,txn_id,order_id,operation_type,status,psp,psp_reference,sku,country,currency,amount_local,fx_rate_applied,fx_date_applied,amount_usd,created_at_utc,captured_at_utc
0,TXN-107955,ORD-507781,SALE,settled,dlocal,DL-90001309,sub_monthly,MX,MXN,129.00,0.055082,2026-05-29,7.11,2026-05-31T23:59:50Z,2026-06-01T00:05:00Z
1,TXN-100884,ORD-500884,SALE,settled,paypal_us,7FTLSR4XAT5TKVLKF,sub_monthly,US,USD,9.99,1.000000,2026-06-01,9.99,2026-06-01T07:03:11Z,2026-06-01T07:12:11Z
2,TXN-106445,ORD-506445,SALE,settled,dlocal,DL-90000750,sub_monthly,CO,COP,29900.00,0.000251,2026-06-01,7.50,2026-06-01T07:11:11Z,2026-06-01T08:27:11Z
3,TXN-106099,ORD-506099,SALE,settled,dlocal,DL-90000404,sub_monthly,MX,MXN,129.00,0.055014,2026-06-01,7.10,2026-06-01T07:11:24Z,2026-06-01T08:17:24Z
4,TXN-105905,ORD-505905,SALE,settled,dlocal,DL-90000210,sub_quarterly,BR,BRL,89.90,0.184519,2026-06-01,16.59,2026-06-01T07:12:27Z,2026-06-01T08:53:27Z


In [16]:
def describe_cols(df, max_unique=20):
      for c in df.columns:
          s = df[c]
          line = f"{c:<18} nulls {s.isna().sum():>5}  unique {s.nunique():>5}  "
          if s.nunique() <= max_unique:
              line += str(s.value_counts(dropna=False).to_dict())
          else:
              line += "e.g. " + ", ".join(s.dropna().head(3))
          print(line)

In [17]:
describe_cols(log)

txn_id             nulls     0  unique  7988  e.g. TXN-107955, TXN-100884, TXN-106445
order_id           nulls     0  unique  7813  e.g. ORD-507781, ORD-500884, ORD-506445
operation_type     nulls     0  unique     3  {'SALE': 7813, 'REFUND': 173, 'CHARGEBACK': 2}
status             nulls     0  unique     3  {'settled': 7425, 'declined': 562, 'pending': 1}
psp                nulls     0  unique     5  {'paypal_us': 2166, 'paypal_eu': 1891, 'adyen': 1792, 'dlocal': 1319, 'google_play': 820}
psp_reference      nulls   820  unique  7168  e.g. DL-90001309, 7FTLSR4XAT5TKVLKF, DL-90000750
sku                nulls     0  unique     4  {'sub_monthly': 4476, 'sub_quarterly': 1567, 'sub_annual': 1166, 'coach_addon': 779}
country            nulls     0  unique    12  {'US': 2841, 'DE': 1337, 'FR': 752, 'BR': 725, 'GB': 708, 'MX': 403, 'NL': 366, 'ES': 273, 'IT': 186, 'CL': 154, 'CO': 131, 'AR': 112}
currency           nulls     0  unique     8  {'EUR': 2914, 'USD': 2841, 'BRL': 725, 'GBP': 708, 

In [18]:
paypal_us = con.sql("select * from raw.raw_paypal_us_activity").df()
print(paypal_us.shape)
paypal_us.head()

(2167, 13)


,Date,Time,Time Zone,Name,Transaction ID,Reference Txn ID,Type,Status,Currency,Gross,Fee,Net,Invoice ID
0,06/01/2026,07:12:11,GMT,Taylor Brooks,7FTLSR4XAT5TKVLKF,NaN,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-500884
1,06/01/2026,07:54:32,GMT,Morgan Hayes,7R82A7SLSXJAD72J1,NaN,Website Payment,Completed,USD,59.99,-2.58,57.41,ORD-500132
2,06/01/2026,07:59:18,GMT,Drew Bennett,S1VNZT7ACC5E6HNYD,NaN,Website Payment,Completed,USD,59.99,-2.58,57.41,ORD-500591
3,06/01/2026,08:03:29,GMT,Riley Carter,BTDR7VQTQ8G4GTMGS,NaN,Website Payment,Completed,USD,9.99,-0.84,9.15,ORD-500479
4,06/01/2026,08:16:21,GMT,Casey Nguyen,NFQZ142PMP4P1BC2Q,NaN,Website Payment,Completed,USD,24.99,-1.36,23.63,ORD-500005


In [19]:
describe_cols(paypal_us)

Date               nulls     0  unique    33  e.g. 06/01/2026, 06/01/2026, 06/01/2026
Time               nulls     0  unique  2117  e.g. 07:12:11, 07:54:32, 07:59:18
Time Zone          nulls     0  unique     1  {'GMT': 2167}
Name               nulls     0  unique    10  {'Jamie Ross': 236, 'Morgan Hayes': 230, 'Taylor Brooks': 225, 'Sam Porter': 225, 'Alex Morgan': 223, 'Drew Bennett': 215, 'Riley Carter': 215, 'Avery Collins': 212, 'Casey Nguyen': 193, 'Jordan Lee': 193}
Transaction ID     nulls     0  unique  2166  e.g. 7FTLSR4XAT5TKVLKF, 7R82A7SLSXJAD72J1, S1VNZT7ACC5E6HNYD
Reference Txn ID   nulls  2110  unique    57  e.g. S1VNZT7ACC5E6HNYD, YM66U594B2K7VTQUD, 1MB89YKKNXT3ALD3W
Type               nulls     0  unique     2  {'Website Payment': 2110, 'Refund': 57}
Status             nulls     0  unique     3  {'Completed': 1942, 'Denied': 168, 'Refunded': 57}
Currency           nulls     0  unique     1  {'USD': 2167}
Gross              nulls     0  unique    11  {'9.99': 1181, '24.